# Chapter 1 ASIC Baseline Evaluation Review

This notebook visualizes the saved ASIC Chapter 1 baseline evaluation package inside the HPC bundle. It does **not** retrain models. All tables and figures below are read from saved artifacts under `artifacts/chapter1/evaluation/asic/baselines/primary_medians`.


## 1. Setup / Paths


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

if "MPLCONFIGDIR" not in os.environ:
    mplconfigdir = Path("/tmp") / "chapter1_mortality_decomposition_matplotlib"
    mplconfigdir.mkdir(parents=True, exist_ok=True)
    os.environ["MPLCONFIGDIR"] = str(mplconfigdir)

try:
    from IPython import get_ipython
    ipython_shell = get_ipython()
    if ipython_shell is None:
        raise RuntimeError("Plain Python execution: use fallback display helpers.")
    from IPython.display import Markdown, display
except Exception:
    ipython_shell = None
    def Markdown(text: str) -> str:
        return text

    def display(obj: object) -> None:
        print(obj)

import matplotlib
if ipython_shell is None:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt


def looks_like_repo_root(path: Path) -> bool:
    return (path / "src" / "chapter1_mortality_decomposition").exists() and (
        (path / "pyproject.toml").exists() or (path / "requirements.txt").exists()
    )


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    search_roots = [start, *start.parents]
    for candidate in search_roots:
        if looks_like_repo_root(candidate):
            return candidate
    for anchor in search_roots:
        try:
            child_dirs = [child for child in anchor.iterdir() if child.is_dir()]
        except Exception:
            continue
        for child in child_dirs:
            if looks_like_repo_root(child):
                return child
    return start


WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = find_repo_root(WORKING_DIRECTORY)
EVAL_ROOT = REPO_ROOT / "artifacts" / "chapter1" / "evaluation" / "asic" / "baselines" / "primary_medians"
MODELS = ["logistic_regression", "xgboost"]
HORIZONS = [8, 16, 24, 48, 72]
PRIMARY_HORIZON = 24

COMBINED_METRICS_PATH = EVAL_ROOT / "combined_metrics.csv"
REPORTING_SPLIT_SUMMARY_PATH = EVAL_ROOT / "reporting_split_summary.csv"
COMBINED_RISK_SUMMARY_PATH = EVAL_ROOT / "combined_risk_binned_summary.csv"
INTERPRETATION_NOTE_PATH = EVAL_ROOT / "interpretation_note.md"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def model_label(model_name: str) -> str:
    return {
        "logistic_regression": "Logistic Regression",
        "xgboost": "XGBoost",
    }.get(model_name, model_name.replace("_", " ").title())


def maybe_read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def evaluation_path(model_name: str, relative_path: str) -> Path:
    return EVAL_ROOT / model_name / relative_path


def show_image_grid(paths: list[Path], titles: list[str], *, ncols: int = 3, figsize_scale: float = 4.5) -> None:
    if not paths:
        display(Markdown("_No figure paths provided._"))
        return
    nrows = int(np.ceil(len(paths) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).reshape(-1)
    for ax, path, title in zip(axes, paths, titles):
        ax.set_title(title)
        if path.exists():
            ax.imshow(plt.imread(path))
            ax.axis("off")
        else:
            ax.axis("off")
            ax.text(0.5, 0.5, f"Missing figure\n{path.name}", ha="center", va="center")
    for ax in axes[len(paths):]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


display(Markdown(f"Working directory: `{WORKING_DIRECTORY}`"))
display(Markdown(f"Repository root: `{REPO_ROOT}`"))
display(Markdown(f"Evaluation root: `{EVAL_ROOT}`"))
display(Markdown(f"Artifacts available: `{EVAL_ROOT.exists()}`"))


## 2. Load Prediction Outputs


In [ ]:
combined_metrics = maybe_read_csv(COMBINED_METRICS_PATH)
reporting_split_summary = maybe_read_csv(REPORTING_SPLIT_SUMMARY_PATH)
combined_risk_summary = maybe_read_csv(COMBINED_RISK_SUMMARY_PATH)
interpretation_note = INTERPRETATION_NOTE_PATH.read_text() if INTERPRETATION_NOTE_PATH.exists() else "_Missing interpretation note._"

if not combined_metrics.empty and not reporting_split_summary.empty:
    reporting_metrics = combined_metrics.merge(
        reporting_split_summary[["model_name", "horizon_h", "selected_split", "selection_reason", "selected_split_evaluable"]],
        left_on=["model_name", "horizon_h", "split"],
        right_on=["model_name", "horizon_h", "selected_split"],
        how="inner",
    )
else:
    reporting_metrics = pd.DataFrame()

display(Markdown("### Reporting Split Selection"))
display(reporting_split_summary)

display(Markdown("### Combined Metric Rows"))
display(combined_metrics.head(12))


## 3. Discrimination Metrics Summary


In [ ]:
if reporting_metrics.empty:
    display(Markdown("_Reporting metrics are missing._"))
else:
    discrimination_table = reporting_metrics[["model_name", "horizon_h", "selected_split", "sample_count", "event_count", "auroc", "auprc", "metric_notes"]].sort_values(["model_name", "horizon_h"])
    display(discrimination_table)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    for model_name in MODELS:
        subset = reporting_metrics[reporting_metrics["model_name"].eq(model_name)].sort_values("horizon_h")
        axes[0].plot(subset["horizon_h"], subset["auroc"], marker="o", linewidth=2, label=model_label(model_name))
        axes[1].plot(subset["horizon_h"], subset["auprc"], marker="o", linewidth=2, label=model_label(model_name))
    axes[0].set_title("AUROC by Horizon")
    axes[1].set_title("AUPRC by Horizon")
    for ax in axes:
        ax.set_xlabel("Horizon (h)")
        ax.set_xticks(HORIZONS)
        ax.set_ylim(0.0, 1.0)
        ax.grid(alpha=0.25)
        ax.legend()
    axes[0].set_ylabel("AUROC")
    axes[1].set_ylabel("AUPRC")
    plt.tight_layout()
    plt.show()


## 4. Calibration Summary


In [ ]:
if reporting_metrics.empty:
    display(Markdown("_Reporting metrics are missing._"))
else:
    calibration_table = reporting_metrics[["model_name", "horizon_h", "selected_split", "calibration_intercept", "calibration_slope", "brier_score", "metric_notes"]].sort_values(["model_name", "horizon_h"])
    display(calibration_table)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
    metric_specs = [
        ("calibration_slope", "Calibration slope", None),
        ("calibration_intercept", "Calibration intercept", None),
        ("brier_score", "Brier score", (0.0, 0.15)),
    ]
    for axis, (column, label, ylim) in zip(axes, metric_specs):
        for model_name in MODELS:
            subset = reporting_metrics[reporting_metrics["model_name"].eq(model_name)].sort_values("horizon_h")
            axis.plot(subset["horizon_h"], subset[column], marker="o", linewidth=2, label=model_label(model_name))
        axis.set_title(label)
        axis.set_xlabel("Horizon (h)")
        axis.set_xticks(HORIZONS)
        axis.grid(alpha=0.25)
        if ylim is not None:
            axis.set_ylim(*ylim)
    axes[0].legend()
    plt.tight_layout()
    plt.show()


## 5. Reliability Plots


In [ ]:
for model_name in MODELS:
    display(Markdown(f"### {model_label(model_name)}"))
    image_paths = [evaluation_path(model_name, f"horizon_{horizon_h}h/reliability_plot.png") for horizon_h in HORIZONS]
    titles = [f"{horizon_h}h" for horizon_h in HORIZONS]
    show_image_grid(image_paths, titles, ncols=3, figsize_scale=4.2)


## 6. Mortality-vs-Risk Plots


In [ ]:
for model_name in MODELS:
    display(Markdown(f"### {model_label(model_name)}"))
    image_paths = [evaluation_path(model_name, f"horizon_{horizon_h}h/mortality_vs_risk_plot.png") for horizon_h in HORIZONS]
    titles = [f"{horizon_h}h" for horizon_h in HORIZONS]
    show_image_grid(image_paths, titles, ncols=3, figsize_scale=4.2)

display(Markdown("### Saved Reporting-Split Risk-Binned Summary"))
display(combined_risk_summary.head(20))


## 7. Horizon Comparison


In [ ]:
for model_name in MODELS:
    display(Markdown(f"### {model_label(model_name)}"))
    horizon_metrics = maybe_read_csv(evaluation_path(model_name, "horizon_comparison_metrics.csv"))
    display(horizon_metrics)
    show_image_grid(
        [
            evaluation_path(model_name, "horizon_comparison_plot.png"),
            evaluation_path(model_name, "horizon_risk_structure_grid.png"),
        ],
        ["Metric comparison", "Mortality-vs-risk across horizons"],
        ncols=2,
        figsize_scale=5.0,
    )


## 8. Site-Stratified Sanity Check


In [ ]:
for model_name in MODELS:
    display(Markdown(f"### {model_label(model_name)} primary {PRIMARY_HORIZON}h site summary"))
    site_summary = maybe_read_csv(evaluation_path(model_name, f"primary_{PRIMARY_HORIZON}h_site_summary.csv"))
    display(site_summary)
    show_image_grid(
        [
            evaluation_path(model_name, f"primary_{PRIMARY_HORIZON}h_site_overview.png"),
            evaluation_path(model_name, f"primary_{PRIMARY_HORIZON}h_site_risk_structure.png"),
        ],
        ["Site overview", "Site risk structure"],
        ncols=2,
        figsize_scale=5.0,
    )


## 9. Short Interpretation Note / Markdown Summary


In [ ]:
display(Markdown(interpretation_note))
